# YOLO11s + LoRA 微调冒烟测试

此 Notebook 使用 Tencent YOLO-Master 的真实 LoRA 参数，在已经审计的数据集上运行 3 epochs。输出仅用于技术验证，**不得作为最终客户演示模型**。

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录，请从项目内启动 Notebook')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('项目根目录:', PROJECT_ROOT)

In [ ]:
import torch
import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT

LORA_KEYS = ('lora_r', 'lora_alpha', 'lora_dropout', 'lora_gradient_checkpointing')
missing = [key for key in LORA_KEYS if key not in DEFAULT_CFG_DICT]
print('Python:', sys.version.split()[0])
print('Ultralytics:', ultralytics.__version__, ultralytics.__file__)
print('CUDA 可用:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if missing:
    raise RuntimeError(f'当前不是支持 LoRA 的 YOLO-Master 环境，缺少参数: {missing}')

In [ ]:
from training.audit_dataset import audit

DATASET_DIR = PROJECT_ROOT / 'data/processed'
DATA_YAML = DATASET_DIR / 'data.yaml'
audit_report = audit(DATASET_DIR)
if not audit_report['passed']:
    raise RuntimeError('数据审计失败，禁止训练：\n' + '\n'.join(audit_report['errors'][:20]))
audit_report['splits']

## 训练参数
RTX 5060 Laptop 8GB 默认使用 batch=8。如果仍显存不足，将 `BATCH` 改为 4。

In [ ]:
MODEL = 'yolo11s.pt'
EPOCHS = 3
IMAGE_SIZE = 640
BATCH = 8
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
OUTPUT_PROJECT = PROJECT_ROOT / 'models/trained/smoke'
RUN_NAME = 'yolo11s_lora_notebook_stable_smoke'
SEED = 42

print('输出目录:', OUTPUT_PROJECT / RUN_NAME)
print('警告：这是 smoke-only 权重，不能部署为最终客户演示模型。')

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    patience=3,
    seed=SEED,
    deterministic=True,
    amp=False,
    pretrained=True,
    cos_lr=True,
    warmup_epochs=5,
    warmup_bias_lr=0.01,
    lora_r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    lora_freeze_bn=True,
    lora_use_rslora=False,
    lora_init_lora_weights='gaussian',
    lora_lr_mult=0.5,
    lora_alpha_warmup=0,
    lora_layer_decay=1.0,
    lora_ortho_weight=0.0,
    lora_dropout_end=0.05,
    lora_dropout_start_ratio=1.0,
    lora_gradient_checkpointing=False,
    project=str(OUTPUT_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

In [ ]:
BEST_PT = OUTPUT_PROJECT / RUN_NAME / 'weights/best.pt'
ADAPTER_DIR = OUTPUT_PROJECT / RUN_NAME / 'lora_adapter'
if not BEST_PT.is_file():
    raise FileNotFoundError(f'训练未生成 best.pt: {BEST_PT}')
if hasattr(model, 'save_lora_only'):
    model.save_lora_only(str(ADAPTER_DIR))
print('best.pt:', BEST_PT)
print('LoRA adapter:', ADAPTER_DIR)
print('SMOKE ONLY — 不得放入 models/deployed。')

In [ ]:
smoke_model = YOLO(str(BEST_PT))
metrics = smoke_model.val(
    data=str(DATA_YAML), split='test', imgsz=640, batch=8, workers=0, device=DEVICE, plots=True,
    project=str(PROJECT_ROOT / 'models/metrics/evaluation_runs'),
    name='smoke_lora_notebook', exist_ok=True,
)
{
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'mAP50': float(metrics.box.map50),
    'mAP50-95': float(metrics.box.map),
}

## 下一步
Notebook 全部通过只表示 LoRA 技术链路可用。最终模型必须通过项目脚本分别完成 150 epochs 完整微调和 LoRA 微调、内部测试集比较及正式导出。